# Hyperion Economy Simulation Notebook

Dieses Notebook nutzt die vollständige Hyperion-Ökonomielogik aus `trade_sim.py` und führt eine **nicht-interaktive Mehrjahres-Simulation** aus.

Ablauf:
1. Parameter in einer Zelle setzen
2. Simulation über festgelegte Perioden/Jahre laufen lassen
3. Ergebnisse als **Pandas-Tabellen** und **Matplotlib-Grafiken** ausgeben


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from trade_sim import HyperionEconomySim, SimulationConfig, GOODS

plt.style.use('ggplot')


## 1) Parameter
Passe diese Werte an und führe die Zelle aus.


In [ ]:
# Simulationsparameter
YEARS = 30
SEED = 7
EVENT_CHANCE = 0.45
TRADE_INTENSITY = 1.0
FACTION_STRENGTH = 1.0

TOP_N_WORLDS = 5
SHOW_TRADE_LOG_PER_YEAR = 3  # Einträge pro Jahr in der Jahrestabelle


## 2) Simulation ausführen


In [ ]:
config = SimulationConfig(
    ticks=YEARS,
    seed=SEED,
    event_chance=EVENT_CHANCE,
    trade_intensity=TRADE_INTENSITY,
    faction_strength=FACTION_STRENGTH,
)
sim = HyperionEconomySim(config)

yearly_rows = []
world_rows = []
price_rows = []

for _ in range(YEARS):
    sim.step()

    yearly_rows.append({
        'year': sim.tick,
        'events': '; '.join(sim.current_events) if sim.current_events else 'keine',
        'core_signal': sim.faction_state['core_signal'],
        'ouster_threat': sim.faction_state['ouster_threat'],
        'templar_access': sim.faction_state['templar_access'],
        'trade_log_excerpt': ' | '.join(sim.trade_log[:SHOW_TRADE_LOG_PER_YEAR]) if sim.trade_log else 'kein Handel',
    })

    for w in sim.worlds:
        world_rows.append({
            'year': sim.tick,
            'world': w.name,
            'category': w.category,
            'faction': w.faction,
            'farcaster': w.farcaster,
            'periphery': w.periphery,
            'stability': w.stability,
            'prosperity': w.prosperity,
            'time_debt': w.time_debt,
            'hyperion_special': w.hyperion_special,
        })
        for good in GOODS:
            price_rows.append({
                'year': sim.tick,
                'world': w.name,
                'good': good,
                'price': w.prices[good],
                'stock': w.stock[good],
            })

df_yearly = pd.DataFrame(yearly_rows)
df_world = pd.DataFrame(world_rows)
df_price = pd.DataFrame(price_rows)

print(f'Fertig. Simulierte Jahre: {YEARS}')
print(f'Datensätze: yearly={len(df_yearly)}, world={len(df_world)}, price={len(df_price)}')


## 3) Tabellen-Auswertung


In [ ]:
# Jahresüberblick
df_yearly.head(10)


In [ ]:
# Letztes Simulationsjahr: Top-Welten nach Wohlstand
last_year = df_world['year'].max()
top_worlds_last = (
    df_world[df_world['year'] == last_year]
    .sort_values(['prosperity', 'stability'], ascending=False)
    .head(TOP_N_WORLDS)
)
top_worlds_last


In [ ]:
# Durchschnittspreise pro Gut und Jahr über alle Welten
df_price_avg = (
    df_price.groupby(['year', 'good'], as_index=False)
    .agg(avg_price=('price', 'mean'), avg_stock=('stock', 'mean'))
)
df_price_avg.head(12)


In [ ]:
# Hyperion-Fokus: Preise und Lager auf Hyperion
df_hyperion = df_price[df_price['world'] == 'Hyperion'].copy()
df_hyperion.head(12)


## 4) Grafiken


In [ ]:
# 4.1 Wohlstandsentwicklung der Top-Welten (letztes Jahr)
top_world_names = top_worlds_last['world'].tolist()
plot_df = df_world[df_world['world'].isin(top_world_names)]

fig, ax = plt.subplots(figsize=(10, 5))
for world_name, grp in plot_df.groupby('world'):
    ax.plot(grp['year'], grp['prosperity'], label=world_name)
ax.set_title('Wohlstandsentwicklung der Top-Welten')
ax.set_xlabel('Jahr')
ax.set_ylabel('Prosperity')
ax.legend()
plt.show()


In [ ]:
# 4.2 Fraktionsdruck über die Zeit
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_yearly['year'], df_yearly['core_signal'], label='Core-Signal')
ax.plot(df_yearly['year'], df_yearly['ouster_threat'], label='Ouster-Druck')
ax.plot(df_yearly['year'], df_yearly['templar_access'], label='Templar-Zugang')
ax.set_title('Fraktions-/Systemdruck je Jahr')
ax.set_xlabel('Jahr')
ax.set_ylabel('Index')
ax.legend()
plt.show()


In [ ]:
# 4.3 Durchschnittspreise pro Gut
pivot_prices = df_price_avg.pivot(index='year', columns='good', values='avg_price')
fig, ax = plt.subplots(figsize=(12, 6))
pivot_prices.plot(ax=ax)
ax.set_title('Durchschnittspreise je Gut (alle Welten)')
ax.set_xlabel('Jahr')
ax.set_ylabel('Preis')
plt.show()


In [ ]:
# 4.4 Hyperion-Sonderwelt: Reliktpreis + Time Debt
hyperion_world = df_world[df_world['world'] == 'Hyperion']
hyperion_relic = df_hyperion[df_hyperion['good'] == 'relikte']

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(hyperion_relic['year'], hyperion_relic['price'], color='purple', label='Reliktpreis Hyperion')
ax1.set_xlabel('Jahr')
ax1.set_ylabel('Reliktpreis', color='purple')

ax2 = ax1.twinx()
ax2.plot(hyperion_world['year'], hyperion_world['time_debt'], color='black', linestyle='--', label='Time Debt Hyperion')
ax2.set_ylabel('Time Debt', color='black')

ax1.set_title('Hyperion: Reliktökonomie vs. Time Debt')
plt.show()


## 5) Exportoption (optional)
Falls du die Daten weiter analysieren möchtest, kannst du sie als CSV speichern.


In [ ]:
# Optional aktivieren:
# df_yearly.to_csv('hyperion_yearly.csv', index=False)
# df_world.to_csv('hyperion_world.csv', index=False)
# df_price.to_csv('hyperion_price.csv', index=False)
